In [ ]:
# %load_ext autoreload
# %autoreload 2


import importlib
import random

### My libraries (not pip)

try:
    import mathlib as Mathlib   #< c++ version (much faster training). Must be compiled first
except ImportError:
    import Mathlib
import Activation
import Loss
import Optimizer
import Accuracy
import Neuron
import Layer
import Batch

# importlib.reload(Mathlib)    #< cpp library
# importlib.reload(Activation)
# importlib.reload(Loss)
# importlib.reload(Optimizer)
# importlib.reload(Accuracy)
# importlib.reload(Neuron)
# importlib.reload(Layer)
# importlib.reload(Batch)

from Neuron import Neuron
from Layer import Layer
from Batch import Batch


#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
  </div>

</div>



#### Go to `proofs_math/Loss` for all Loss proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Loss/SoftmaxCrossEntropyDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/Loss/LossDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>

#### Go to `proofs_math/Neuron` for all Neuron proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Neuron/NeuronDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>



## Function that will be used to create datasets

In [3]:
def createInputs(inputCount=10):
    return([Mathlib.randomNumber(-10, 10, 2) for _ in range(inputCount)])

## Functions to run a forward and backward pass on a super simple network with two layer totaling 9 neurons

In [4]:
random.seed(1)

inputsCount = 5
target = 2
inputs = createInputs(inputsCount)

layer1NeuronCount = 4
layer2NeuronCount = 5

layer1 = Layer(inputsCount, layer1NeuronCount, activationFunc=Activation.ReLU)         #< weights/biases determine neuron count
layer2 = Layer(layer1NeuronCount, layer2NeuronCount, activationFunc=Activation.ReLU)   #< weights/biases determine neuron count

accuracyCalculator = Accuracy.Accuracy_hard()

def stepForward():
    layer1Output = layer1.forward(inputs)
    layer2Output = layer2.forward(layer1Output)
    softmaxOutput = Activation.Softmax.forward(layer2Output)
    error = Loss.Entropy.forwardSparse(softmaxOutput, target)
    accuracy = accuracyCalculator.epoch(softmaxOutput, target)    #< we want to maximize the 2nd output
    return(layer1Output, layer2Output, softmaxOutput, error, accuracy)

def stepBackward(softmaxOutput):
    d_error  = Loss.Entropy.backwardSparse(softmaxOutput, target)
    d_softmax = Activation.Softmax.backward(softmaxOutput)
    d_crossEntropy = Mathlib.dotVectorMatrix(d_error, d_softmax) #< not that we can calculate the cross entropy function much simpler using the shortcut backwardCrossEntropy, but I choose to take the long way in this example because its easier to understand
    d_layer2 = layer2.backward(d_crossEntropy)
    d_layer1 = layer1.backward(d_layer2)
    return(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

def printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy):
    print(f"INPUTS: {inputs}\n    This layer should have {inputsCount} inputs")
    print(f"WEIGHTS: {layer1.getWeights()}\n    this layer should have {layer1NeuronCount} weights with {inputsCount} values in them")
    print(f"BIASES: {layer1.getBiases()}\n    this layer should have {layer1NeuronCount} biases")
    print(f"RESULT: {layer1Output}\n     this layer should have {layer1NeuronCount} outputs")
    print()
    print(f"WEIGHTS: {layer1.getWeights()}\n    this layer should have {layer2NeuronCount} weights with {len(layer1Output)} values in them")
    print(f"BIASES: {layer1.getBiases()}\n    this layer should have {layer2NeuronCount} biases")
    print(f"RESULT: {layer2Output}\n     this layer should have {layer2NeuronCount} outputs")
    print("SOFTMAX: ", softmaxOutput)
    print("TARGET: ", target)
    print(f"ERROR: {error} should be between 0 and 16.12")
    print("ACCURACY: ", accuracy)

def printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1):
    print("d_ERROR: ", d_error)
    print(f"d_SOFTMAX: {d_softmax}\n    this layer should be {layer2NeuronCount}x{len(d_softmax)}")
    print(f"d_CROSS_ENTROPY: {d_crossEntropy}\n    this layer should have {layer2NeuronCount} outputs")
    print("d_LAYER 2: ", d_layer2)
    print("d_LAYER 1: ", d_layer1)

layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy)
d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

INPUTS: [-9.970000267028809, 1.2699999809265137, -6.130000114440918, 6.170000076293945, 1.7000000476837158]
    This layer should have 5 inputs
WEIGHTS: [[-0.03999999910593033, -0.27000001072883606, 0.7099999785423279, 0.5799999833106995, 0.4399999976158142], [-0.5799999833106995, 0.6399999856948853, 0.3799999952316284, 0.019999999552965164, -0.3499999940395355], [-0.8700000047683716, -0.7300000190734863, -0.23999999463558197, -0.6299999952316284, -0.6000000238418579], [0.8700000047683716, -0.10000000149011612, -0.6800000071525574, -0.8899999856948853, -0.8799999952316284]]
    this layer should have 4 weights with 5 values in them
BIASES: [0.0, 0.0, 0.0, 0.0]
    this layer should have 4 biases
RESULT: [0.030199885368347168, 3.7943999767303467, 4.3109002113342285, 0.0]
     this layer should have 4 outputs

WEIGHTS: [[-0.03999999910593033, -0.27000001072883606, 0.7099999785423279, 0.5799999833106995, 0.4399999976158142], [-0.5799999833106995, 0.6399999856948853, 0.3799999952316284, 0.

### Optimizing the neural network

In [5]:
optimizer = Optimizer.Optimizer_SGD(learning_rate=0.0001)
for i in range(5000):
    layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
    d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
    optimizer.update(layer2)
    optimizer.update(layer1)
    print(f"Epoch {i+1}: Error: {error}, accuracy: {accuracy}")

Epoch 1: Error: 7.134974482824222, accuracy: 0.0
Epoch 2: Error: 7.100230121713767, accuracy: 0.0
Epoch 3: Error: 7.065545372783732, accuracy: 0.0
Epoch 4: Error: 7.030922033580868, accuracy: 0.0
Epoch 5: Error: 6.997307274384433, accuracy: 0.0
Epoch 6: Error: 6.966418513711901, accuracy: 0.0
Epoch 7: Error: 6.935587859428722, accuracy: 0.0
Epoch 8: Error: 6.904814717196468, accuracy: 0.0
Epoch 9: Error: 6.874099334655543, accuracy: 0.0
Epoch 10: Error: 6.843442254075731, accuracy: 0.0
Epoch 11: Error: 6.812845018267817, accuracy: 0.0
Epoch 12: Error: 6.782304994792365, accuracy: 0.0
Epoch 13: Error: 6.751824068627359, accuracy: 0.0
Epoch 14: Error: 6.721401427559317, accuracy: 0.0
Epoch 15: Error: 6.691034429122647, accuracy: 0.0
Epoch 16: Error: 6.660729802905511, accuracy: 0.0
Epoch 17: Error: 6.630481994843245, accuracy: 0.0
Epoch 18: Error: 6.600293010417355, accuracy: 0.0
Epoch 19: Error: 6.570161048216759, accuracy: 0.0
Epoch 20: Error: 6.540088311445432, accuracy: 0.0
Epoch 21:

In [7]:
import json
with open("datasets/squareData.json", "r") as f:
    a = json.load(f)
    X = a["X"]         #< 32x32 image 1 = pixel, 0 = no pixel
    y = a["y"]

print("Converting data to hilbert space")
for i in range(len(X)):
    X[i] = Mathlib.hilbertFlatten(X[i])   #< map to 1D using hilbert space (hilbert is used here so the resolution has minimal effect)
    print(f"{i}/{len(X)}", end="\r")
print(f"\rDone/{len(X)}")


try:
    with open("trainedModel.json", "r") as f:
        savedModel = json.load(f)
    spiralLayer1 = Batch(inputCount=32*32, neuronCount=32, activationFunc=Activation.LeakyReLU)
    spiralLayer2 = Batch(inputCount=32, neuronCount=2, activationFunc=Activation.Pass)

    spiralLayer1.weights = savedModel["layer1"]["weights"]
    spiralLayer1.biases = savedModel["layer1"]["biases"]
    spiralLayer2.weights = savedModel["layer2"]["weights"]
    spiralLayer2.biases = savedModel["layer2"]["biases"]
    print("Successfully loaded 'trainedModel.json'!")

except FileNotFoundError:
    print("Error: 'trainedModel.json' not found. Training from scratch.")
    spiralLayer1 = Batch(
    # spiralLayer1 = Layer(
                        inputCount=32*32,
                        neuronCount=32,   #< output features
                        activationFunc=Activation.LeakyReLU)
    spiralLayer2 = Batch(
    # spiralLayer2 = Layer(
                        inputCount=32,
                        neuronCount=2,
                        activationFunc=Activation.Pass)
softmaxCrossEntropy = Loss.SoftmaxCrossEntropy()
spiralOptimizer = Optimizer.Optimizer_SGD(learning_rate=0.3)
accuracyCalculator = Accuracy.Accuracy_hard()

for epoch in range(1_000):
    BATCH_SIZE = 32
    numSamples = len(X)

    for i in range(0, numSamples, BATCH_SIZE):
        X_batch = X[i : i + BATCH_SIZE]
        y_batch = y[i : i + BATCH_SIZE]

        ### Forwards pass
        ### layer1 -> layer2 -> layer3 -> softmax_error
        layer1Output = spiralLayer1.forward(X_batch)                              #< returns a list of lists of outputs
        layer2Output = spiralLayer2.forward(layer1Output)                         #< returns a list of lists of outputs
        # print(layer2Output)
        # error = softmaxCrossEntropy.forward(layer2Output, y_batch)              #< returns mean error (layer mode)
        error = softmaxCrossEntropy.forward_batch(layer2Output, y_batch)          #< returns mean error (batch mode)
        accuracy = accuracyCalculator.epoch(layer2Output, y_batch)                #< returns accuracy of model %right

        ### Backwards pass
        ### error_softmax -> layer2 ->  layer1 
        d_crossEntropy = softmaxCrossEntropy.backward_batch()
        # d_crossEntropy = softmaxCrossEntropy.backward()
        d_layer2 = spiralLayer2.backward(d_crossEntropy)
        # print([Mathlib.mean(d) for d in d_layer2])
        d_layer1 = spiralLayer1.backward(d_layer2)
        # print([Mathlib.mean(d) for d in d_layer1])

        spiralOptimizer.update(spiralLayer2)
        spiralOptimizer.update(spiralLayer1)

    # if epoch%10 == 0:
    print(f"Epoch {epoch}: Error: {error}, accuracy: {accuracy}")
    
    ## Save model
    modelState = {
        "layer1": {
            "weights": spiralLayer1.getWeights(),
            "biases": spiralLayer1.getBiases()
        },
        "layer2": {
            "weights": spiralLayer2.getWeights(),
            "biases": spiralLayer2.getBiases()
        }
    }

    with open("trainedModel.json", "w") as f:
        json.dump(modelState, f, indent=4)


Converting data to hilbert space
Done/1000
Successfully loaded 'trainedModel.json'!
Epoch 0: Error: 0.10327461361885071, accuracy: 0.812
Epoch 1: Error: 0.10323585569858551, accuracy: 0.813
Epoch 2: Error: 0.10318800806999207, accuracy: 0.8133333333333334
Epoch 3: Error: 0.10314202308654785, accuracy: 0.8135
Epoch 4: Error: 0.10310013592243195, accuracy: 0.8136
Epoch 5: Error: 0.10304994881153107, accuracy: 0.8136666666666666
Epoch 6: Error: 0.10300005972385406, accuracy: 0.8137142857142857
Epoch 7: Error: 0.10295036435127258, accuracy: 0.81375
Epoch 8: Error: 0.10289661586284637, accuracy: 0.8137777777777778
Epoch 9: Error: 0.10284194350242615, accuracy: 0.8138
Epoch 10: Error: 0.10278723388910294, accuracy: 0.8139090909090909
Epoch 11: Error: 0.102733314037323, accuracy: 0.814
Epoch 12: Error: 0.10267801582813263, accuracy: 0.814076923076923
Epoch 13: Error: 0.10262102633714676, accuracy: 0.8141428571428572
Epoch 14: Error: 0.1025610864162445, accuracy: 0.8142666666666667
Epoch 15: E

KeyboardInterrupt: 

The model takes slightly less than 1.5 hours to train to 77% accuracy

In [4]:
import Demo
import tkinter as tk
root = tk.Tk()
app = Demo.DrawingApp(root)
root.mainloop()